# 4. Vanna Training

This notebook trains Vanna with the schema and sample queries for the EMR database.

In [1]:
import os
import sys

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.utils import get_vanna

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("1. Initializing Vanna...")
# Ensure PostgreSQL is running and notebook 3 has been executed
vn = get_vanna()

1. Initializing Vanna...


In [3]:
print("2. Training Vanna with DDL...")
ddl = """
CREATE TABLE emr_records (
    emr_name VARCHAR,
    machine_model VARCHAR,
    model_family VARCHAR,
    serial_number VARCHAR,
    branch_site VARCHAR,
    account VARCHAR,
    sub_call_type VARCHAR,
    pmact_type VARCHAR,
    subjects TEXT,
    symptom TEXT,
    caused_of_problem TEXT,
    techcare_component VARCHAR,
    techcare_sub_component VARCHAR,
    main_cause_part_no VARCHAR,
    part_description VARCHAR,
    created_date TIMESTAMP,
    closed_date TIMESTAMP,
    cluster_id INTEGER,
    cluster_prob FLOAT,
    cluster_label VARCHAR
);
"""
vn.train(ddl=ddl)

2. Training Vanna with DDL...
Adding ddl: 
CREATE TABLE emr_records (
    emr_name VARCHAR,
    machine_model VARCHAR,
    model_family VARCHAR,
    serial_number VARCHAR,
    branch_site VARCHAR,
    account VARCHAR,
    sub_call_type VARCHAR,
    pmact_type VARCHAR,
    subjects TEXT,
    symptom TEXT,
    caused_of_problem TEXT,
    techcare_component VARCHAR,
    techcare_sub_component VARCHAR,
    main_cause_part_no VARCHAR,
    part_description VARCHAR,
    created_date TIMESTAMP,
    closed_date TIMESTAMP,
    cluster_id INTEGER,
    cluster_prob FLOAT,
    cluster_label VARCHAR
);



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4381.01it/s]


'c9166b1c-cc72-5254-a3b5-c5dd9602b2e4-ddl'

In [4]:
print("3. Training Vanna with Documentation...")
doc = """
The 'emr_records' table contains Equipment Maintenance Records for heavy machinery.
- 'machine_model' is the specific equipment model (e.g., PC200-8).
- 'model_family' is the broader category (e.g., PC200).
- 'branch_site' is the location where the equipment is operated.
- 'cluster_label' categorizes the text description of the fault into a common type (e.g., 'Hydraulic Leak').
- Use 'created_date' when filtering by month or year.
"""
vn.train(documentation=doc)

3. Training Vanna with Documentation...
Adding documentation....


'b8b17c72-8050-534e-8f94-2531448136fd-doc'

In [5]:
print("4. Training Vanna with Sample Q&A...")

qa_pairs = [
    ("Berapa total fault per model?", "SELECT machine_model, COUNT(*) as total FROM emr_records GROUP BY machine_model ORDER BY total DESC;"),
    ("Model apa yang paling sering rusak?", "SELECT machine_model, COUNT(*) as total FROM emr_records GROUP BY machine_model ORDER BY total DESC LIMIT 1;"),
    ("Tampilkan 5 site dengan masalah terbanyak", "SELECT branch_site, COUNT(*) as total FROM emr_records GROUP BY branch_site ORDER BY total DESC LIMIT 5;"),
    ("Berapa jumlah kerusakan berdasarkan kategori (cluster)?", "SELECT cluster_label, COUNT(*) as total FROM emr_records GROUP BY cluster_label ORDER BY total DESC;"),
    ("Tampilkan tren kerusakan per bulan", "SELECT DATE_TRUNC('month', created_date) as month, COUNT(*) as total FROM emr_records GROUP BY month ORDER BY month;"),
    ("Tampilkan tren masalah pada PC200 per bulan", "SELECT DATE_TRUNC('month', created_date) as month, COUNT(*) as total FROM emr_records WHERE model_family = 'PC200' GROUP BY month ORDER BY month;"),
    ("Sebutkan top 3 masalah (cluster) di site Jakarta", "SELECT cluster_label, COUNT(*) as total FROM emr_records WHERE branch_site ILIKE '%Jakarta%' GROUP BY cluster_label ORDER BY total DESC LIMIT 3;"),
    ("berikan perbandingan antara unit yang sering bermasalah dengan yang paling tidak sering bermasalah", "(SELECT machine_model, COUNT(*) as total_faults, 'Paling Sering' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults DESC LIMIT 5) UNION ALL (SELECT machine_model, COUNT(*) as total_faults, 'Paling Jarang' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults ASC LIMIT 5) ORDER BY status DESC, total_faults DESC;"),
    ("perbandingan unit paling sering rusak dengan unit yang paling jarang rusak", "(SELECT machine_model, COUNT(*) as total_faults, 'Paling Sering' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults DESC LIMIT 5) UNION ALL (SELECT machine_model, COUNT(*) as total_faults, 'Paling Jarang' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults ASC LIMIT 5) ORDER BY status DESC, total_faults DESC;")
]

for q, sql in qa_pairs:
    vn.train(question=q, sql=sql)

print("Training complete.")

4. Training Vanna with Sample Q&A...
Training complete.


In [ ]:
print("5. Testing SQL Generation...")
# This will use the local Ollama LLM defined in our custom Vanna implementation
test_query = "Berapa banyak kerusakan hydraulic pada PC200?"
sql = vn.generate_sql(test_query)
print(f"Question: {test_query}\nGenerated SQL: {sql}")

if sql:
    df = vn.run_sql(sql)
    print(df)

5. Testing SQL Generation...
Question: Berapa banyak kerusakan hydraulic pada PC200?
Generated SQL: SELECT COUNT(*) as total FROM emr_records WHERE model_family = 'PC200' AND cluster_label ILIKE '%hydraulic%';
   total
0     71


: 